# 00 arXiv Suche

Dieses Notebook ruft vier arXiv-API-Suchen ab, zeigt pro Suche die Trefferzahlen und exportiert die gefundenen Paper als Excel-Datei.

Export enthält:
- `query_counts`: Anzahl laut arXiv und Anzahl geladener Einträge
- `all_results`: alle Treffer inklusive Query-Zuordnung
- `unique_papers`: deduplizierte Paper nach arXiv-ID
- je ein Sheet pro Query

In [12]:
from pathlib import Path
import importlib.util
from urllib.parse import urlparse, parse_qs
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import time
import xml.etree.ElementTree as ET

import pandas as pd

if importlib.util.find_spec("openpyxl") is None:
    raise ImportError("Für den Excel-Export bitte einmal installieren: %pip install openpyxl")

pd.set_option("display.max_colwidth", 120)

OUTPUT_DIR = Path("../runs/arxiv_search")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_PATH = OUTPUT_DIR / "arxiv_search_results.xlsx"
REQUEST_SLEEP_SECONDS = 10
RETRY_HTTP_STATUS_CODES = {429, 500, 502, 503, 504}
RETRY_SLEEP_SECONDS = [20, 60, 120, 240]
CONTACT_EMAIL = None  # Optional: "deine.mail@example.com"
EXPORT_PATH

PosixPath('../runs/arxiv_search/arxiv_search_results.xlsx')

In [13]:
QUERIES = {
    "llm_agents_planning": "https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22language%20agent%22%20OR%20all:%22large%20language%20model%20agent%22%29%20AND%20%28all:%22planning%22%20OR%20all:%22reasoning%22%20OR%20all:%22tool%20use%22%20OR%20all:%22autonomous%20agent%22%29%20AND%20submittedDate:%5B202201010000%20TO%20202612312359%5D%29&start=0&max_results=200&sortBy=submittedDate&sortOrder=descending",
    "web_agents_llm": "https://export.arxiv.org/api/query?search_query=%28%28all:%22web%20agent%22%20OR%20all:%22web%20automation%22%20OR%20all:%22web%20navigation%22%29%20AND%20%28all:%22large%20language%20model%22%20OR%20all:%22LLM%22%20OR%20all:%22language%20agent%22%29%20AND%20submittedDate:%5B202201010000%20TO%20202612312359%5D%29&start=0&max_results=200&sortBy=submittedDate&sortOrder=descending",
    "replanning_reflection": "https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22language%20agent%22%20OR%20all:%22web%20agent%22%29%20AND%20%28all:%22replanning%22%20OR%20all:%22reflection%22%20OR%20all:%22verification%22%20OR%20all:%22validation%22%20OR%20all:%22feedback%22%29%20AND%20submittedDate:%5B202201010000%20TO%20202612312359%5D%29&start=0&max_results=200&sortBy=submittedDate&sortOrder=descending",
    "benchmarks_efficiency": "https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22web%20agent%22%29%20AND%20%28all:%22success%20rate%22%20OR%20all:%22benchmark%22%20OR%20all:%22token%20cost%22%20OR%20all:%22latency%22%20OR%20all:%22runtime%22%20OR%20all:%22efficiency%22%29%20AND%20submittedDate:%5B202201010000%20TO%20202612312359%5D%29&start=0&max_results=200&sortBy=submittedDate&sortOrder=descending",
}

list(QUERIES)

['llm_agents_planning',
 'web_agents_llm',
 'replanning_reflection',
 'benchmarks_efficiency']

In [14]:
NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "opensearch": "http://a9.com/-/spec/opensearch/1.1/",
    "arxiv": "http://arxiv.org/schemas/atom",
}


def text_or_none(node, path, ns=NS):
    found = node.find(path, ns)
    if found is None or found.text is None:
        return None
    return " ".join(found.text.split())


def arxiv_id_from_url(entry_id):
    if not entry_id:
        return None
    return entry_id.rstrip("/").split("/")[-1]


def search_query_from_url(query_url):
    return parse_qs(urlparse(query_url).query).get("search_query", [None])[0]


def parse_entry(entry, query_name, query_url, query_string):
    entry_id = text_or_none(entry, "atom:id")
    published = text_or_none(entry, "atom:published")
    updated = text_or_none(entry, "atom:updated")
    authors = [text_or_none(author, "atom:name") for author in entry.findall("atom:author", NS)]
    categories = [cat.attrib.get("term") for cat in entry.findall("atom:category", NS)]
    links = entry.findall("atom:link", NS)
    pdf_url = next((link.attrib.get("href") for link in links if link.attrib.get("title") == "pdf"), None)
    abs_url = next((link.attrib.get("href") for link in links if link.attrib.get("rel") == "alternate"), entry_id)
    doi = text_or_none(entry, "arxiv:doi")
    primary_category = entry.find("arxiv:primary_category", NS)

    return {
        "query_name": query_name,
        "query_string": query_string,
        "arxiv_id": arxiv_id_from_url(entry_id),
        "title": text_or_none(entry, "atom:title"),
        "authors": "; ".join(author for author in authors if author),
        "year": int(published[:4]) if published else None,
        "published": published,
        "updated": updated,
        "abstract": text_or_none(entry, "atom:summary"),
        "primary_category": primary_category.attrib.get("term") if primary_category is not None else None,
        "categories": "; ".join(cat for cat in categories if cat),
        "doi": doi,
        "abs_url": abs_url,
        "pdf_url": pdf_url,
        "query_url": query_url,
    }


def fetch_xml_with_retries(query_url):
    user_agent = "masterarbeit-arxiv-search/1.0"
    if CONTACT_EMAIL:
        user_agent += f" (mailto:{CONTACT_EMAIL})"
    request = Request(query_url, headers={"User-Agent": user_agent})

    for attempt in range(len(RETRY_SLEEP_SECONDS) + 1):
        try:
            with urlopen(request, timeout=60) as response:
                return response.read()
        except HTTPError as exc:
            retry_after = exc.headers.get("Retry-After")
            wait_seconds = int(retry_after) if retry_after and retry_after.isdigit() else RETRY_SLEEP_SECONDS[min(attempt, len(RETRY_SLEEP_SECONDS) - 1)]
            if exc.code not in RETRY_HTTP_STATUS_CODES or attempt == len(RETRY_SLEEP_SECONDS):
                raise
            print(f"  arXiv HTTP {exc.code}. Warte {wait_seconds}s und versuche es erneut ...")
            time.sleep(wait_seconds)
        except URLError:
            if attempt == len(RETRY_SLEEP_SECONDS):
                raise
            wait_seconds = RETRY_SLEEP_SECONDS[min(attempt, len(RETRY_SLEEP_SECONDS) - 1)]
            print(f"  Netzwerkfehler. Warte {wait_seconds}s und versuche es erneut ...")
            time.sleep(wait_seconds)

    raise RuntimeError("arXiv API konnte nach mehreren Versuchen nicht geladen werden.")


def fetch_arxiv_query(query_name, query_url):
    xml_bytes = fetch_xml_with_retries(query_url)

    root = ET.fromstring(xml_bytes)
    total_results = int(text_or_none(root, "opensearch:totalResults") or 0)
    start_index = int(text_or_none(root, "opensearch:startIndex") or 0)
    items_per_page = int(text_or_none(root, "opensearch:itemsPerPage") or 0)
    entries = root.findall("atom:entry", NS)

    query_string = search_query_from_url(query_url)
    rows = [parse_entry(entry, query_name, query_url, query_string) for entry in entries]
    meta = {
        "query_name": query_name,
        "total_results_arxiv": total_results,
        "loaded_results": len(rows),
        "start_index": start_index,
        "items_per_page": items_per_page,
        "max_results_in_url": int(parse_qs(urlparse(query_url).query).get("max_results", [0])[0]),
        "query_string": query_string,
        "query_url": query_url,
    }
    return meta, rows

In [15]:
RESULT_COLUMNS = [
    "query_name", "query_string", "arxiv_id", "title", "authors", "year", "published", "updated",
    "abstract", "primary_category", "categories", "doi", "abs_url", "pdf_url", "query_url",
]

all_meta = []
all_rows = []

for i, (query_name, query_url) in enumerate(QUERIES.items(), start=1):
    print(f"[{i}/{len(QUERIES)}] Lade {query_name} ...")
    try:
        meta, rows = fetch_arxiv_query(query_name, query_url)
        meta["status"] = "ok"
        meta["error"] = None
        all_meta.append(meta)
        all_rows.extend(rows)
        print(f"  arXiv totalResults={meta['total_results_arxiv']}, geladen={meta['loaded_results']}")
    except Exception as exc:
        query_string = search_query_from_url(query_url)
        all_meta.append({
            "query_name": query_name,
            "total_results_arxiv": None,
            "loaded_results": 0,
            "start_index": None,
            "items_per_page": None,
            "max_results_in_url": int(parse_qs(urlparse(query_url).query).get("max_results", [0])[0]),
            "query_string": query_string,
            "query_url": query_url,
            "status": "error",
            "error": repr(exc),
        })
        print(f"  Fehler bei {query_name}: {exc!r}")
    if i < len(QUERIES):
        time.sleep(REQUEST_SLEEP_SECONDS)

counts_df = pd.DataFrame(all_meta)
results_df = pd.DataFrame(all_rows, columns=RESULT_COLUMNS)

counts_df

[1/4] Lade llm_agents_planning ...
  arXiv totalResults=1754, geladen=200
[2/4] Lade web_agents_llm ...
  arXiv totalResults=221, geladen=200
[3/4] Lade replanning_reflection ...
  arXiv totalResults=1229, geladen=200
[4/4] Lade benchmarks_efficiency ...
  arXiv totalResults=1864, geladen=200


,query_name,total_results_arxiv,loaded_results,start_index,items_per_page,max_results_in_url,query_string,query_url,status,error
0,llm_agents_planning,1754,200,0,200,200,"((all:""LLM agent"" OR all:""language agent"" OR all:""large language model agent"") AND (all:""planning"" OR all:""reasoning...",https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22language%20agent%22%20OR%2...,ok,None
1,web_agents_llm,221,200,0,200,200,"((all:""web agent"" OR all:""web automation"" OR all:""web navigation"") AND (all:""large language model"" OR all:""LLM"" OR a...",https://export.arxiv.org/api/query?search_query=%28%28all:%22web%20agent%22%20OR%20all:%22web%20automation%22%20OR%2...,ok,None
2,replanning_reflection,1229,200,0,200,200,"((all:""LLM agent"" OR all:""language agent"" OR all:""web agent"") AND (all:""replanning"" OR all:""reflection"" OR all:""veri...",https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22language%20agent%22%20OR%2...,ok,None
3,benchmarks_efficiency,1864,200,0,200,200,"((all:""LLM agent"" OR all:""web agent"") AND (all:""success rate"" OR all:""benchmark"" OR all:""token cost"" OR all:""latency...",https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22web%20agent%22%29%20AND%20...,ok,None


## Überschneidungen nach Suchstring-Reihenfolge

Diese Auswertung nimmt die Reihenfolge aus `QUERIES`: Suchstring 1 ist die Basis. Für Suchstring 2 wird gezählt, wie viele Paper schon in Suchstring 1 waren. Für Suchstring 3 wird gezählt, wie viele Paper schon in Suchstring 1 oder 2 waren, usw.

In [19]:
seen_ids = set()
seen_by_query = {}
overlap_summary_rows = []
overlap_detail_rows = []

for query_position, query_name in enumerate(QUERIES, start=1):
    query_df = results_df.loc[results_df["query_name"] == query_name].copy()
    query_ids = set(query_df["arxiv_id"].dropna())
    duplicate_ids = query_ids & seen_ids
    new_ids = query_ids - seen_ids

    overlap_summary_rows.append({
        "query_position": query_position,
        "query_name": query_name,
        "loaded_results": len(query_df),
        "unique_in_this_query": len(query_ids),
        "already_seen_in_previous_queries": len(duplicate_ids),
        "new_vs_previous_queries": len(new_ids),
        "previous_queries_checked": "; ".join(list(QUERIES)[:query_position - 1]),
    })

    for arxiv_id in sorted(duplicate_ids):
        paper_row = query_df.loc[query_df["arxiv_id"] == arxiv_id].iloc[0]
        previous_queries = [previous_query for previous_query, ids in seen_by_query.items() if arxiv_id in ids]
        overlap_detail_rows.append({
            "current_query_position": query_position,
            "current_query_name": query_name,
            "already_seen_in_queries": "; ".join(previous_queries),
            "arxiv_id": arxiv_id,
            "title": paper_row.get("title"),
            "authors": paper_row.get("authors"),
            "year": paper_row.get("year"),
            "published": paper_row.get("published"),
            "abs_url": paper_row.get("abs_url"),
            "pdf_url": paper_row.get("pdf_url"),
        })

    seen_by_query[query_name] = query_ids
    seen_ids |= query_ids

OVERLAP_DUPLICATE_COLUMNS = [
    "current_query_position", "current_query_name", "already_seen_in_queries", "arxiv_id",
    "title", "authors", "year", "published", "abs_url", "pdf_url",
]

overlap_summary_df = pd.DataFrame(overlap_summary_rows)
overlap_duplicates_df = pd.DataFrame(overlap_detail_rows, columns=OVERLAP_DUPLICATE_COLUMNS)

counts_df = counts_df.drop(
    columns=["already_seen_in_previous_queries", "new_vs_previous_queries"],
    errors="ignore",
)
counts_df = counts_df.merge(
    overlap_summary_df[["query_name", "already_seen_in_previous_queries", "new_vs_previous_queries"]],
    on="query_name",
    how="left",
)

overlap_summary_df

,query_position,query_name,loaded_results,unique_in_this_query,already_seen_in_previous_queries,new_vs_previous_queries,previous_queries_checked
0,1,llm_agents_planning,200,200,0,200,
1,2,web_agents_llm,200,200,3,197,llm_agents_planning
2,3,replanning_reflection,200,200,83,117,llm_agents_planning; web_agents_llm
3,4,benchmarks_efficiency,200,200,147,53,llm_agents_planning; web_agents_llm; replanning_reflection


In [18]:
if results_df.empty:
    unique_df = pd.DataFrame(columns=[col for col in RESULT_COLUMNS if col != "query_name"] + ["matched_queries"])
else:
    unique_df = (
        results_df
        .sort_values(["published", "query_name"], ascending=[False, True])
        .drop_duplicates(subset=["arxiv_id"], keep="first")
        .copy()
    )

    query_memberships = (
        results_df
        .groupby("arxiv_id")["query_name"]
        .apply(lambda values: "; ".join(sorted(set(values))))
        .rename("matched_queries")
    )
    unique_df = unique_df.drop(columns=["query_name"]).merge(query_memberships, on="arxiv_id", how="left")

print(f"Alle Query-Treffer: {len(results_df)}")
print(f"Deduplizierte Paper: {len(unique_df)}")

unique_df.head(10)

Alle Query-Treffer: 800
Deduplizierte Paper: 567


,query_string,arxiv_id,title,authors,year,published,updated,abstract,primary_category,categories,doi,abs_url,pdf_url,query_url,matched_queries
0,"((all:""LLM agent"" OR all:""web agent"") AND (all:""success rate"" OR all:""benchmark"" OR all:""token cost"" OR all:""latency...",2605.22759v1,Towards a General Intelligence and Interface for Wearable Health Data,Girish Narayanswamy; Maxwell A. Xu; A. Ali Heydari; Samy Abdel-Ghaffar; Marius Guerard; Kara Vaillancourt; Zhihan Zh...,2026,2026-05-21T17:24:06Z,2026-05-21T17:24:06Z,"While ubiquitous wearable sensors capture a wealth of behavioral and physiological information, effectively transfor...",cs.AI,cs.AI,NaN,https://arxiv.org/abs/2605.22759v1,https://arxiv.org/pdf/2605.22759v1,https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22web%20agent%22%29%20AND%20...,benchmarks_efficiency; replanning_reflection
1,"((all:""LLM agent"" OR all:""web agent"") AND (all:""success rate"" OR all:""benchmark"" OR all:""token cost"" OR all:""latency...",2605.22721v1,Self-Evolving Multi-Agent Systems via Decentralized Memory,Guangya Hao; Yunbo Long; Zhuokai Zhao,2026,2026-05-21T16:55:40Z,2026-05-21T16:55:40Z,Self-evolving multi-agent systems (MAS) have emerged as a promising route to LLM agents that continually improve fro...,cs.MA,cs.MA,NaN,https://arxiv.org/abs/2605.22721v1,https://arxiv.org/pdf/2605.22721v1,https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22web%20agent%22%29%20AND%20...,benchmarks_efficiency; replanning_reflection
2,"((all:""LLM agent"" OR all:""web agent"") AND (all:""success rate"" OR all:""benchmark"" OR all:""token cost"" OR all:""latency...",2605.22664v1,WorkstreamBench: Evaluating LLM Agents on End-to-End Spreadsheet Tasks in Finance,Thomson Yen; Julian Poeltl; Harshith Srinivas Gear; Yilin Meng; Joshua Fan; Adam Shen; Yili Liu; Ali Bauyrzhan; Siri...,2026,2026-05-21T16:06:34Z,2026-05-21T16:06:34Z,"LLM agents are increasingly expected to carry out end-to-end workflows, producing complete artifacts from high-level...",cs.AI,cs.AI,NaN,https://arxiv.org/abs/2605.22664v1,https://arxiv.org/pdf/2605.22664v1,https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22web%20agent%22%29%20AND%20...,benchmarks_efficiency; replanning_reflection
3,"((all:""LLM agent"" OR all:""web agent"") AND (all:""success rate"" OR all:""benchmark"" OR all:""token cost"" OR all:""latency...",2605.22608v1,Agentic CLEAR: Automating Multi-Level Evaluation of LLM Agents,Asaf Yehudai; Lilach Eden; Michal Shmueli-Scheuer,2026,2026-05-21T15:26:02Z,2026-05-21T15:26:02Z,"Agentic systems are becoming more capable: agents define strategies, take actions, and interact with different envir...",cs.CL,cs.CL; cs.AI,NaN,https://arxiv.org/abs/2605.22608v1,https://arxiv.org/pdf/2605.22608v1,https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22web%20agent%22%29%20AND%20...,benchmarks_efficiency; replanning_reflection
4,"((all:""LLM agent"" OR all:""web agent"") AND (all:""success rate"" OR all:""benchmark"" OR all:""token cost"" OR all:""latency...",2605.22566v1,GraphFlow: A Graph-Based Workflow Management for Efficient LLM-Agent Serving,Ao Li; Shangpeng Yang; Fahao Chen; Tianheng Xu; Peng Li; Zhou Su,2026,2026-05-21T14:45:40Z,2026-05-21T14:45:40Z,Large Language Model (LLM)-based agents demonstrate strong reasoning and execution capabilities on complex tasks whe...,cs.LG,cs.LG,NaN,https://arxiv.org/abs/2605.22566v1,https://arxiv.org/pdf/2605.22566v1,https://export.arxiv.org/api/query?search_query=%28%28all:%22LLM%20agent%22%20OR%20all:%22web%20agent%22%29%20AND%20...,benchmarks_efficiency; llm_agents_planning
5,"((all:""LLM agent"" OR all:""web agent"") AND (all:""success rate"" OR all:""benchmark"" OR all:""token cost"" OR all:""latency...",2605.22411v1,DeferMem: Query-Time Evidence Distillation via Reinforcement Learning for Long-Term Memory QA,Jianing Yin; Tan Tang,2026,202

In [17]:
with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
    counts_df.to_excel(writer, sheet_name="query_counts", index=False)
    overlap_summary_df.to_excel(writer, sheet_name="overlap_summary", index=False)
    overlap_duplicates_df.to_excel(writer, sheet_name="overlap_duplicates", index=False)
    results_df.to_excel(writer, sheet_name="all_results", index=False)
    unique_df.to_excel(writer, sheet_name="unique_papers", index=False)

    for query_name in QUERIES:
        sheet_name = query_name[:31]
        results_df.loc[results_df["query_name"] == query_name].to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Exportiert nach: {EXPORT_PATH.resolve()}")

Exportiert nach: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/arxiv_search/arxiv_search_results.xlsx


Hinweis: Die URLs enthalten `max_results=200`. Wenn `total_results_arxiv` größer als `loaded_results` ist, lädt dieses Notebook bewusst nur die ersten 200 Treffer pro Query. Für vollständiges Paging müsste man `start` in 200er-Schritten erhöhen.